# Preprocessing: Spatial Data

Baut eine neue CSV pro Listing mit:
- Koordinaten (`latitude`, `longitude`)
- kleinerer Stadtbezirk (`neighbourhood`) und grober Zone (`neighbourhood_group`), per **Point-in-Polygon Spatial Join** aus den Koordinaten und `neighbourhoods.geojson` bestimmt
- Preis (`price`), aus `listings.csv` (Hinweis: `calendar.csv` enthaelt entgegen der Annahme keine Preise - die `price`-Spalte dort ist durchgaengig leer, siehe Check unten)

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

## 1. Listings laden (id, Koordinaten, Preis)

In [2]:
listings = pd.read_csv(
    "listings.csv",
    usecols=["id", "latitude", "longitude", "price"]
)

listings["price"] = (
    listings["price"]
    .replace({r"\$": "", r",": ""}, regex=True)
    .astype(float)
)

print(listings.shape)
listings.head()

(8215, 4)


,id,latitude,longitude,price
0,49287,37.398980,-5.995330,93.0
1,108236,37.396858,-5.999127,120.0
2,111140,37.395924,-5.999317,89.0
3,159596,37.386710,-5.995810,NaN
4,179629,37.387310,-5.990950,146.0


## 2. Check: Preise in calendar.csv?

Die `price`-Spalte in `calendar.csv` ist komplett leer (auch bei `available == 't'`). Preise kommen daher ausschliesslich aus `listings.csv`.

In [3]:
calendar_price_check = pd.read_csv("calendar.csv", usecols=["available", "price"])
print("Zeilen gesamt:", len(calendar_price_check))
print("Nicht-leere Preise:", calendar_price_check["price"].notna().sum())
del calendar_price_check

Zeilen gesamt: 2998475
Nicht-leere Preise: 0


## 3. Stadtteil-Grenzen laden

In [4]:
districts = gpd.read_file("neighbourhoods.geojson")[
    ["neighbourhood", "neighbourhood_group", "geometry"]
]

print(districts.shape)
districts.head()

(108, 3)


,neighbourhood,neighbourhood_group,geometry
0,Los Remedios,Los Remedios,"MULTIPOLYGON (((-5.99314 37.37508, -5.9929 37...."
1,Heliópolis,Palmera - Bellavista,"MULTIPOLYGON (((-5.98038 37.3568, -5.97843 37...."
2,Bda. de Pineda,Palmera - Bellavista,"MULTIPOLYGON (((-5.95876 37.35578, -5.95932 37..."
3,El Plantinar,Sur,"MULTIPOLYGON (((-5.97509 37.37718, -5.97497 37..."
4,Juan XXIII,Cerro - Amate,"MULTIPOLYGON (((-5.94357 37.37838, -5.9436 37...."


## 4. Spatial Join: Listings (Punkte) -> Stadtteile (Polygone)

Fuer jedes Listing wird anhand von `latitude`/`longitude` bestimmt, in welchem Polygon (`neighbourhood`) es liegt. `neighbourhood_group` ergibt sich direkt mit.

In [5]:
listings_gdf = gpd.GeoDataFrame(
    listings,
    geometry=[Point(xy) for xy in zip(listings["longitude"], listings["latitude"])],
    crs="EPSG:4326",
)

assert listings_gdf.crs == districts.crs

joined = gpd.sjoin(listings_gdf, districts, how="left", predicate="within")
joined = joined.drop(columns=["index_right", "geometry"])

print("Listings ohne Stadtteil-Treffer:", joined["neighbourhood"].isna().sum())
joined.head()

Listings ohne Stadtteil-Treffer: 0


,id,latitude,longitude,price,neighbourhood,neighbourhood_group
0,49287,37.398980,-5.995330,93.0,San Lorenzo,Casco Antiguo
1,108236,37.396858,-5.999127,120.0,San Vicente,Casco Antiguo
2,111140,37.395924,-5.999317,89.0,San Vicente,Casco Antiguo
3,159596,37.386710,-5.995810,NaN,Arenal,Casco Antiguo
4,179629,37.387310,-5.990950,146.0,Santa Cruz,Casco Antiguo


## 5. Ergebnis zusammenstellen und speichern

In [6]:
result = joined[
    ["id", "latitude", "longitude", "neighbourhood_group", "neighbourhood", "price"]
].rename(columns={
    "neighbourhood_group": "district_group",
    "neighbourhood": "district",
})

result.to_csv("listings_spatial.csv", index=False)
print(result.shape)
result.head()

(8215, 6)


,id,latitude,longitude,district_group,district,price
0,49287,37.398980,-5.995330,Casco Antiguo,San Lorenzo,93.0
1,108236,37.396858,-5.999127,Casco Antiguo,San Vicente,120.0
2,111140,37.395924,-5.999317,Casco Antiguo,San Vicente,89.0
3,159596,37.386710,-5.995810,Casco Antiguo,Arenal,NaN
4,179629,37.387310,-5.990950,Casco Antiguo,Santa Cruz,146.0
